# Head Writing Analysis

Analyze what attention heads write to the residual stream: directions,
logit effects, consistency, residual contribution, and output rank.

## Why This Matters

Attention head writing reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_writing_analysis import (
    head_writing_directions, head_logit_writing,
    head_writing_consistency, head_residual_contribution,
    head_writing_rank,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Writing Directions

What direction does each head write into the residual stream?

In [ ]:
result = head_writing_directions(model, tokens, layer=0)
for h in result['per_head']:
    bar = '█' * int(min(h['output_norm'] * 20, 20))
    print(f"  Head {h['head']}: out_norm={h['output_norm']:.4f}, "
          f"mean_norm={h['mean_output_norm']:.4f} {bar}")
print('\nDirection pairs:')
for p in result['direction_pairs']:
    print(f"  H{p['head_a']} vs H{p['head_b']}: cos={p['cosine']:.4f}")

## Logit Writing

What tokens does each head's output promote/suppress?

In [ ]:
result = head_logit_writing(model, tokens, layer=0)
for h in result['per_head']:
    print(f"  Head {h['head']}: promotes tok {h['top_promoted_token']:3d} "
          f"({h['top_promoted_logit']:+.4f}), suppresses tok {h['top_suppressed_token']:3d} "
          f"({h['top_suppressed_logit']:+.4f}), range={h['logit_range']:.4f}")

## Writing Consistency

Does each head write in a consistent direction across positions?

In [ ]:
for layer in range(4):
    result = head_writing_consistency(model, tokens, layer=layer)
    for h in result['per_head']:
        flag = ' [CONSISTENT]' if h['is_consistent'] else ''
        print(f"  L{layer}H{h['head']}: consistency={h['mean_direction_consistency']:.4f}, "
              f"norm={h['mean_output_norm']:.4f}{flag}")

## Residual Contribution

How much does each head contribute to (or work against) the residual stream?

In [ ]:
result = head_residual_contribution(model, tokens)
for h in result['per_head']:
    sign = '+' if h['is_constructive'] else '-'
    print(f"  L{h['layer']}H{h['head']}: [{sign}] norm={h['output_norm']:.4f}, "
          f"alignment={h['residual_alignment']:.4f}")

## Writing Rank

Effective rank of each head's output across positions.

In [ ]:
for layer in range(4):
    result = head_writing_rank(model, tokens, layer=layer)
    for h in result['per_head']:
        flag = ' [LOW-RANK]' if h['is_low_rank'] else ''
        print(f"  L{layer}H{h['head']}: rank={h['effective_rank']:.2f}, "
              f"top_sv={h['top_sv_fraction']:.3f}{flag}")